# DeepWeeds — Chạy trên Colab GPU T4

File này chạy độc lập trên Colab, không cần clone repo.

**Cách dùng:**
1. Colab > Runtime > Change runtime type > Hardware accelerator = **GPU (T4)**
2. Runtime > Run all
3. Link ảnh dùng trong file: `https://zenodo.org/records/7939060/files/images.zip?download=1`


In [ ]:
# 0. Kiểm tra GPU T4 + cài thư viện
!nvidia-smi -L
%pip -q install timm scikit-learn pandas matplotlib pillow

import torch
print("torch:", torch.__version__, "| cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("KHÔNG CÓ GPU -> Runtime > Change runtime type > T4 GPU rồi chạy lại")


In [ ]:
# 1. Tải ảnh từ link bạn đưa + kiểm tra MD5 + giải nén + tự tìm IMAGES_DIR
import os, hashlib, glob, urllib.request, shutil
from collections import Counter

IMG_URL = "https://zenodo.org/records/7939060/files/images.zip?download=1"
MD5_OK = "b7b30f96d466fba86016aa5a26606e0f"

os.makedirs("data/labels", exist_ok=True)

jpgs_exist = glob.glob("data/**/*.jpg", recursive=True)
if len(jpgs_exist) >= 17000:
    print(f"Đã có sẵn {len(jpgs_exist)} ảnh -> bỏ qua tải Zenodo")
else:
    if not os.path.exists("data/images.zip"):
        print("Đang tải images.zip từ Zenodo (~490MB, chờ vài phút)...")
        # dùng wget của Colab cho nhanh, có progress
        !wget -q --show-progress -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"
    print("Kiểm tra MD5...")
    h = hashlib.md5()
    with open("data/images.zip", "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    print("MD5:", h.hexdigest())
    assert h.hexdigest() == MD5_OK, f"MD5 sai: {h.hexdigest()}"
    print("MD5 OK -> giải nén...")
    !unzip -q -n data/images.zip -d data/
    print("Giải nén xong")

# Tải nhãn + fold chia sẵn từ GitHub tác giả
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    dst = f"data/labels/{name}.csv"
    if not os.path.exists(dst):
        print("Tải", name)
        urllib.request.urlretrieve(f"{BASE}/{name}.csv", dst)
!ls -lh data/labels

# Tự tìm thư mục chứa ảnh (zip có thể bung ra data/images/ hoặc data/*.jpg)
jpgs = glob.glob("data/**/*.jpg", recursive=True)
assert len(jpgs) > 0, "Không thấy .jpg trong data/"
IMAGES_DIR = Counter(os.path.dirname(j) for j in jpgs).most_common(1)[0][0]
LABELS_DIR = "data/labels"
print(f"IMAGES_DIR = {IMAGES_DIR} | tổng jpg = {len(jpgs)}")


In [ ]:
# 2. Kiểm tra split train/val/test (bắt buộc theo README 2.1)
import pandas as pd

tr_df = pd.read_csv(f"{LABELS_DIR}/train_subset0.csv")
va_df = pd.read_csv(f"{LABELS_DIR}/val_subset0.csv")
te_df = pd.read_csv(f"{LABELS_DIR}/test_subset0.csv")
print(tr_df.shape, va_df.shape, te_df.shape)
print(tr_df.head(3).to_string())

s_tr, s_va, s_te = set(tr_df.Filename), set(va_df.Filename), set(te_df.Filename)
print(f"train={len(s_tr)} val={len(s_va)} test={len(s_te)} total={len(s_tr|s_va|s_te)}")
assert len(s_tr & s_va) == 0 and len(s_tr & s_te) == 0 and len(s_va & s_te) == 0, "Giao khác rỗng!"
assert len(s_tr | s_va | s_te) == 17509, "Hợp phải = 17509"
print("Split OK: giao rỗng, hợp đủ 17509")
print(tr_df.Label.value_counts().sort_index().to_string())


In [ ]:
# 3. Dataset + DataLoader dùng luôn trên T4
import random
import numpy as np
from pathlib import Path
from PIL import Image
from torchvision import transforms as T

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMG_SIZE = 224
BATCH_SIZE = 32  # T4 15GB để 32 là an toàn; nếu OOM giảm xuống 16

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(0.5),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class DeepWeeds(torch.utils.data.Dataset):
    def __init__(self, df, images_dir, tf):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        with Image.open(self.images_dir / str(r["Filename"])) as im:
            img = im.convert("RGB")
        if self.tf:
            img = self.tf(img)
        return img, int(r["Label"]), str(r["Filename"])

def make_loader(df, train):
    ds = DeepWeeds(df, IMAGES_DIR, train_tf if train else eval_tf)
    return torch.utils.data.DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=train,
        num_workers=2, pin_memory=True, drop_last=train)

train_loader = make_loader(tr_df, True)
val_loader = make_loader(va_df, False)
test_loader = make_loader(te_df, False)
x, y, fn = next(iter(train_loader))
print("batch:", tuple(x.shape), y[:8].tolist(), fn[:2])


In [ ]:
# 4. Model + train bằng GPU T4 (AMP) — đổi BACKBONE / EPOCHS tùy ý
import timm, math, time
from sklearn.metrics import f1_score

BACKBONE = "resnet50"  # thử: resnet50, convnext_tiny, efficientnet_b0, mobilenetv3_large_100, deit_small_patch16_224
EPOCHS = 5              # chạy thử 5; chạy thật để 10-12
LR_BACKBONE, LR_HEAD = 1e-4, 1e-3

device = torch.device("cuda")
assert torch.cuda.is_available(), "Bắt buộc chạy Runtime GPU"
print("Dùng GPU:", torch.cuda.get_device_name(0))

torch.manual_seed(0); np.random.seed(0); random.seed(0)
model = timm.create_model(BACKBONE, pretrained=True, num_classes=9)
model.to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"{BACKBONE}: {n_params:.1f}M params")

# 3 nhóm LR như slide trang 52: backbone / norm-bias (wd=0) / head mới
head_ids = set(id(p) for p in model.get_classifier().parameters())
g1, g2, g3 = [], [], []
for n, p in model.named_parameters():
    if not p.requires_grad: continue
    if id(p) in head_ids: g3.append(p)
    elif p.ndim <= 1: g2.append(p)
    else: g1.append(p)
opt = torch.optim.AdamW([
    {"params": g1, "lr": LR_BACKBONE, "weight_decay": 0.05},
    {"params": g2, "lr": LR_BACKBONE, "weight_decay": 0.0},
    {"params": g3, "lr": LR_HEAD, "weight_decay": 0.05}])

steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch
warmup = int(1 * steps_per_epoch)
def lr_lambda(s):
    if s < warmup: return (s + 1) / max(warmup, 1)
    t = (s - warmup) / max(total_steps - warmup, 1)
    return 0.5 * (1 + math.cos(math.pi * t))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
sca = torch.cuda.amp.GradScaler()
crit = torch.nn.CrossEntropyLoss()

best_f1, best_state = -1, None
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot_loss = 0
    for xb, yb, _ in train_loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            loss = crit(model(xb), yb)
        sca.scale(loss).backward(); sca.step(opt); sca.update(); sched.step()
        tot_loss += loss.item() * len(xb)
    # val
    model.eval(); ys, ps = [], []
    with torch.inference_mode():
        for xb, yb, _ in val_loader:
            p = model(xb.to(device)).argmax(1).cpu().numpy()
            ps += p.tolist(); ys += yb.numpy().tolist()
    f1 = f1_score(ys, ps, average="macro")
    acc = float(np.mean(np.array(ys) == np.array(ps)))
    print(f"epoch {ep+1}/{EPOCHS} loss={tot_loss/len(tr_df):.4f} val_acc={acc:.4f} val_macroF1={f1:.4f} {time.time()-t0:.0f}s", flush=True)
    if f1 > best_f1: best_f1, best_state = f1, {k: v.cpu() for k, v in model.state_dict().items()}

torch.save({"backbone": BACKBONE, "state": best_state, "val_f1": best_f1}, "best.pt")
print(f"Best val macro-F1 = {best_f1:.4f} -> đã lưu best.pt")


In [ ]:
# 5. Đánh giá test 1 lần + đo độ trễ T4 đúng cách (warmup + synchronize + p50/p95/p99)
import time
from sklearn.metrics import f1_score, accuracy_score

ckpt = torch.load("best.pt", map_location=device)
model.load_state_dict(ckpt["state"]); model.to(device).eval()
print("Nạp best.pt, val F1 lúc train:", round(float(ckpt["val_f1"]), 4))

ys, ps, probs = [], [], []
with torch.inference_mode():
    for xb, yb, _ in test_loader:
        lg = model(xb.to(device))
        pr = lg.softmax(1).cpu().numpy()
        ps += pr.argmax(1).tolist(); ys += yb.numpy().tolist(); probs.append(pr)
print(f"TEST acc={accuracy_score(ys, ps):.4f} macro-F1={f1_score(ys, ps, average='macro'):.4f}")

# Đo latency batch=1 (giống robot) và batch=32
model.eval()
for bs in (1, 32):
    x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=device)
    for _ in range(10):  # warmup bỏ đi
        with torch.inference_mode(): _ = model(x)
    torch.cuda.synchronize()
    ts = []
    for _ in range(100):
        torch.cuda.synchronize(); t0 = time.time()
        with torch.inference_mode(): _ = model(x)
        torch.cuda.synchronize(); ts.append((time.time() - t0) * 1000)
    ts = sorted(ts)
    print(f"batch={bs}: p50={np.percentile(ts,50):.1f} p95={np.percentile(ts,95):.1f} p99={np.percentile(ts,99):.1f} ms")


In [ ]:
# 6. LUU + TAI OUTPUT (bat buoc chay rieng cell nay - Colab mat file neu tat runtime)
import os, zipfile, glob, shutil
os.makedirs("outputs", exist_ok=True)
print("--- file hien co ---")
!ls -lh best.pt outputs/ 2>&1 | head -20
# 6a. luu csv du doan test (dung bien ys/ps/probs tu cell 5)
try:
    import pandas as pd, numpy as np
    ys_a, ps_a = np.array(ys), np.array(ps)
    probs_all = np.concatenate(probs)
    df_out = pd.DataFrame(probs_all, columns=[f"p{i}" for i in range(9)])
    df_out.insert(0, "y_pred", ps_a)
    df_out.insert(0, "y_true", ys_a)
    df_out.insert(0, "Filename", te_df["Filename"].values)
    df_out.to_csv("outputs/predictions_test.csv", index=False)
    print("saved outputs/predictions_test.csv", df_out.shape)
except Exception as e:
    print("bo qua csv:", type(e).__name__, str(e)[:200])
# 6b. zip lai de tai 1 lan cho gon
with zipfile.ZipFile("outputs_colab.zip", "w", compression=zipfile.ZIP_DEFLATED) as z:
    if os.path.exists("best.pt"):
        z.write("best.pt")
    for f in glob.glob("outputs/*"):
        z.write(f)
!ls -lh outputs_colab.zip best.pt outputs/
# 6c. backup ra Drive (neu co mount, tat runtime van con)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    shutil.copy("outputs_colab.zip", "/content/drive/MyDrive/outputs_colab.zip")
    print("da copy ra Drive: /content/drive/MyDrive/outputs_colab.zip")
except Exception as e:
    print("khong copy Drive (chay local hoac chua mount):", str(e)[:150])
# 6d. tai ve may (Colab) - tren local thi file nam san o outputs_colab.zip / outputs/
try:
    from google.colab import files
    files.download("outputs_colab.zip")
    print("dang tai outputs_colab.zip ve may - neu khong thay popup, kiem tra chan popup / Files panel ben trai > chuot phai > Download")
except Exception as e:
    print("khong phai Colab -> file nam o:", os.path.abspath("outputs_colab.zip"))
